# FAST-Sync for seven matrix Lie groups

This example builds self-contained synchronization graphs for every supported group. The noisy `Rot3` and `Pose3` cases receive a closer look and compact visualizations.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/FastSyncExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gtsam

## A reusable graph builder

Every graph below uses arbitrary keys, a three-edge cycle, and one prior. The prior only left-aligns the rounded solution; it does not change the relaxed relative solution.

In [ ]:
def triangle_graph(values, between_factor, prior_factor, dimension):
    keys = (10, 42, 77)
    model = gtsam.noiseModel.Isotropic.Sigma(dimension, 0.1)
    graph = gtsam.NonlinearFactorGraph()
    for i, j in ((0, 1), (1, 2), (2, 0)):
        graph.add(between_factor(keys[i], keys[j], values[i].between(values[j]), model))
    graph.add(prior_factor(keys[0], values[0], model))
    return graph, keys

## Rot2, Pose2, Similarity2, Similarity3, and SL4

In [ ]:
rot2_values = [gtsam.Rot2(a) for a in (0.2, 0.7, -0.4)]
pose2_values = [gtsam.Pose2(1, -2, 0.2), gtsam.Pose2(2, 0.5, 0.7), gtsam.Pose2(-1, 1.5, -0.4)]
sim2_values = [gtsam.Similarity2(rot2_values[i], np.array([i, -0.5*i]), s) for i, s in enumerate((1.1, 0.9, 1.3))]
rot3_small = [gtsam.Rot3.Expmap(np.array([0.04*i, -0.02*i, 0.03*i])) for i in range(3)]
sim3_values = [gtsam.Similarity3(rot3_small[i], np.array([i, 0.2*i, -0.1*i]), s) for i, s in enumerate((1.1, 0.9, 1.3))]
sl4_values = [gtsam.SL4.Expmap(np.linspace(0.001*(i + 1), 0.015*(i + 1), 15)) for i in range(3)]
cases = [
    ('Rot2', rot2_values, gtsam.BetweenFactorRot2, gtsam.PriorFactorRot2, 1, gtsam.fastSyncRot2, lambda v, k: v.atRot2(k)),
    ('Pose2', pose2_values, gtsam.BetweenFactorPose2, gtsam.PriorFactorPose2, 3, gtsam.fastSyncPose2, lambda v, k: v.atPose2(k)),
    ('Similarity2', sim2_values, gtsam.BetweenFactorSimilarity2, gtsam.PriorFactorSimilarity2, 4, gtsam.fastSyncSimilarity2, lambda v, k: v.atSimilarity2(k)),
    ('Similarity3', sim3_values, gtsam.BetweenFactorSimilarity3, gtsam.PriorFactorSimilarity3, 7, gtsam.fastSyncSimilarity3, lambda v, k: v.atSimilarity3(k)),
    ('SL4', sl4_values, gtsam.BetweenFactorSL4, gtsam.PriorFactorSL4, 15, gtsam.fastSyncSL4, lambda v, k: v.atSL4(k)),
]
for name, values, between, prior, dim, solve, at in cases:
    graph, keys = triangle_graph(values, between, prior, dim)
    result = solve(graph)
    determinant = np.linalg.det(at(result, keys[-1]).matrix())
    print(f'{name:11s}: {result.size()} values, det(last)={determinant:.6f}')

## Noisy Rot3 cycle

We perturb every relative rotation deterministically. The plot shows the geodesic orientation error after alignment to the prior.

In [ ]:
rotations = [gtsam.Rot3.Expmap(np.array([0.10*i, -0.04*i, 0.06*i])) for i in range(7)]
rot3_graph = gtsam.NonlinearFactorGraph()
rot3_model = gtsam.noiseModel.Isotropic.Sigma(3, 0.08)
rot3_edges = [(i, i + 1) for i in range(6)] + [(6, 0), (1, 5)]
for edge, (i, j) in enumerate(rot3_edges):
    noise = gtsam.Rot3.Expmap(0.012 * np.array([np.sin(edge), np.cos(edge), (-1)**edge]))
    measured = rotations[i].between(rotations[j]).compose(noise)
    rot3_graph.add(gtsam.BetweenFactorRot3(i, j, measured, rot3_model))
rot3_graph.add(gtsam.PriorFactorRot3(0, rotations[0], rot3_model))
rot3_result = gtsam.fastSyncRot3(rot3_graph)
rot3_error = [np.linalg.norm(gtsam.Rot3.Logmap(rotations[i].between(rot3_result.atRot3(i)))) * 180 / np.pi for i in range(7)]
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.bar(range(7), rot3_error, color='#4c78a8')
ax.set(xlabel='key', ylabel='orientation error (deg)', title='FAST-Sync on a noisy Rot3 graph')
ax.grid(axis='y', alpha=0.25)
plt.show()

## Noisy Pose3 trajectory

FAST-Sync estimates rotation and translation in one ambient linear system. The projected trajectory is already suitable as an initializer for a nonlinear optimizer.

In [ ]:
poses = [gtsam.Pose3(rotations[i], np.array([i, 0.12*i*i, 0.3*np.sin(i)])) for i in range(7)]
pose3_graph = gtsam.NonlinearFactorGraph()
pose3_model = gtsam.noiseModel.Isotropic.Sigma(6, 0.1)
for edge, (i, j) in enumerate(rot3_edges):
    delta = 0.01 * np.array([np.sin(edge), np.cos(edge), (-1)**edge, np.cos(edge), np.sin(edge), 0.5])
    measured = poses[i].between(poses[j]).compose(gtsam.Pose3.Expmap(delta))
    pose3_graph.add(gtsam.BetweenFactorPose3(i, j, measured, pose3_model))
pose3_graph.add(gtsam.PriorFactorPose3(0, poses[0], pose3_model))
pose3_result = gtsam.fastSyncPose3(pose3_graph)
truth_xyz = np.vstack([pose.translation() for pose in poses])
estimate_xyz = np.vstack([pose3_result.atPose3(i).translation() for i in range(7)])
fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(projection='3d')
ax.plot(*truth_xyz.T, 'o-', label='truth', color='#54a24b')
ax.plot(*estimate_xyz.T, 's--', label='FAST-Sync', color='#e45756')
ax.set(xlabel='x', ylabel='y', zlabel='z', title='Noisy Pose3 initialization')
ax.legend()
plt.show()
print('RMS translation error:', np.sqrt(np.mean(np.sum((truth_xyz - estimate_xyz)**2, axis=1))))

The seven entry points above are generated from the single C++ template declaration. They require finite, positive, isotropic between-factor noise and a connected measurement graph. See the [FAST-Sync documentation](../../../gtsam/slam/doc/FastSync.ipynb) for the block solve, gauge convention, and projection details.